In [ ]:
# ============================================================
# Colab: RL for simplifying "hard" unknots (CSV→GCS first-col only)
# ============================================================

# --------- 0) Install dependencies ----------
!pip -q install "gymnasium>=0.29" "stable-baselines3>=2.3.0" spherogram gcsfs numpy tqdm

# --------- 1) Imports & seeds ----------
import os, re, json, random, math, io, gzip, csv, time
import numpy as np
from typing import List, Iterable, Any, Optional
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback
from tqdm.auto import tqdm
import numpy as np

from spherogram import Link
import gcsfs

from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import configure

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# --------- 2) CSV → GCS reader (first column only) ----------
def _normalize_gcs_path(path: str) -> str:
    # gcsfs usually expects "bucket/path". Accept "gs://bucket/path" as well.
    return path[5:] if path.startswith("gs://") else path

def read_first_column_gcs(path: str,
                          max_lines: int | None = None,
                          has_header: bool = True,
                          delimiter: str | None = None,
                          encoding: str = "utf-8") -> list[str]:
    """
    Read a CSV (optionally .gz) from GCS and return only the first column as strings.
    - path: e.g. "gs://bucket/folder/file.csv" or "bucket/folder/file.csv"
    - max_lines: cap rows returned (after header handling)
    - has_header: skip first row if True
    - delimiter: set explicitly ("," or "\t"); if None, sniff a small sample
    - encoding: for non-gz files
    """
    fs = gcsfs.GCSFileSystem(token="anon")
    path = _normalize_gcs_path(path)
    results: list[str] = []

    def make_reader(fh):
        nonlocal delimiter
        if delimiter is None:
            pos = fh.tell()
            sample = fh.read(4096)
            fh.seek(pos)
            try:
                dialect = csv.Sniffer().sniff(sample)
            except Exception:
                dialect = csv.excel  # default (,)
            return csv.reader(fh, dialect)
        else:
            return csv.reader(fh, delimiter=delimiter)

    if path.lower().endswith(".gz"):
        with fs.open(path, "rb") as fb:
            with gzip.GzipFile(fileobj=fb, mode="rb") as gz:
                with io.TextIOWrapper(gz, encoding=encoding, newline="") as ft:
                    reader = make_reader(ft)
                    if has_header:
                        next(reader, None)
                    for i, row in enumerate(reader):
                        if max_lines is not None and i >= max_lines: break
                        if row:
                            results.append(str(row[0]).strip())
    else:
        with fs.open(path, "rt", encoding=encoding, newline="") as f:
            reader = make_reader(f)
            if has_header:
                next(reader, None)
            for i, row in enumerate(reader):
                if max_lines is not None and i >= max_lines: break
                if row:
                    results.append(str(row[0]).strip())
    return results

# --------- 3) Strict PD/DT parser + cleaner ----------
_RE_DT_PREFIX = re.compile(r'^\s*DT\s*:\s*\[', re.I)
_RE_PDLIST = re.compile(r'^\s*\[\s*(\[\s*\d+(?:\s*,\s*\d+){3}\s*\]\s*,?\s*)+\]\s*$') # [[...], ...]
_RE_XPD       = re.compile(r'[Xx]\s*\[')

def parse_link_strict(s: str) -> Link:
    """
    Accept ONLY PD/DT encodings:
      - 'DT: [ ... ]'
      - PD as nested lists: [[8,3,1,4],[2,6,3,5],...]
      - PD in X[...] blocks: X[a,b,c,d], X[...], ...
    Also unwrap simple JSON/JSONL if it contains 'pd' or 'dt' fields.
    """
    t = s.strip()
    if _RE_DT_PREFIX.match(t):
        return Link(t)
    if _RE_PDLIST.match(t):
    # Convert safely into Python object
        try:
            pd_obj = json.loads(t)  # works if it's valid JSON-like
        except json.JSONDecodeError:
            # fallback for python-style lists with no quotes
            import ast
            pd_obj = ast.literal_eval(t)
        return Link(pd_obj)
    if _RE_XPD.search(t):
        try:
            return Link(t)
        except Exception:
            items = re.findall(r'[Xx]\s*\[([^\]]+)\]', t)
            if not items:
                raise
            blocks = []
            for it in items:
                nums = [int(x.strip()) for x in it.split(',')]
                if len(nums) != 4:
                    raise ValueError("PD block must have 4 integers")
                blocks.append(nums)
            return Link(str(blocks))
    if (t.startswith("{") or t.startswith("[")) and not _RE_PDLIST.match(t):
        try:
            obj = json.loads(t)
            for key in ("pd","PD","pd_code","PD_code","dt","DT"):
                if key in obj:
                    return parse_link_strict(obj[key])
        except Exception:
            pass
    raise ValueError("Not a PD/DT code")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

# --------- 4) Spherogram helpers ----------
def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def can_reduce_basic(L: Link) -> bool:
    before = crossings(L)
    tmp = Link(L.PD_code())
    changed = tmp.simplify(mode="basic")
    return bool(changed and crossings(tmp) < before)

# --- UPDATED: pure RIII only (no RI/RII), in-place on the given Link ---
def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    """
    Apply up to k random Type-III moves using Spherogram internals
    without triggering any RI/RII simplification. Returns (link, done).
    """
    # lazy import so we don't change your global imports
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link  # operate in-place
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)  # in-place RIII
            if crossings(L) == c0:  # ensure it's a pure RIII (no crossing change)
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

# --------- 5) RL Environment (macro actions with blocking + simple ranking reward) ----------
# The action is MultiDiscrete: [mode, cap]
#   mode ∈ {0:basic, 1:level, 2:pickup, 3:backtrack}
#   cap  ∈ {0..cap_max}

maxstepsdone = 1000
###################
###################
###################
###################


# 1) Config
@dataclass
class EnvCfg:
    max_steps: int = maxstepsdone
    step_penalty: float = 0.05           # small time cost
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 16
    ##################
    ##################
    ##################
    ##################
    # Simple mode ranking reward: 0 > 1 > 2 > 3
    mode_rewards: tuple[float, float, float, float] = (3.0, 2.0, 1.0, 0.0)

class SphKnotEnv(gym.Env):
    # ... keep everything else as you have it ...

    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)

        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )

        self.L: Optional[Link] = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False

        # NEW: blocking state (True = blocked). Index by mode.
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        """If requested mode is blocked, advance to the next unblocked mode (cyclic)."""
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        # Fallback (shouldn't happen): use backtrack
        return min(3, self.num_actions - 1)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        for _ in range(10):
            s = self.rng.choice(self.pd_lines)
            try:
                self.L = parse_link_strict(s)
                break
            except Exception:
                self.L = None
        if self.L is None:
            self.L = parse_link_strict(self.pd_lines[0])
        return self._obs(), {"crossings": crossings(self.L)}

    def step(self, action):
        # Unpack action
        self._steps += 1
        if isinstance(action, (list, tuple, np.ndarray)):
            mode_req, cap = int(action[0]), int(action[1])
        else:
            mode_req, cap = int(action), 0
        cap = max(0, min(cap, self.cfg.cap_max))

        # Apply blocking: remap to first unblocked mode (cyclic from requested)
        mode = self._map_blocked_mode(mode_req)

        # Context
        c_before = crossings(self.L)

        # Execute chosen (possibly remapped) mode
        if mode == 0:
            self.L.simplify(mode="basic")

        elif mode == 1:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="level", type_III_limit=steps)

        elif mode == 2:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="pickup", type_III_limit=steps)

        elif mode == 3 and self.num_actions == 4:
            steps = (cap if cap > 0 else 1)
            self.L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
            # optional: tiny RIII shuffle is ok to keep or remove; keeping it is fine
            self.L, _ = riii_shuffle_only_link(self.L, min(steps, 2))

        # Post
        c_after = crossings(self.L)
        delta = c_before - c_after  # success iff delta > 0
        self._last_drop = max(delta, 0)

        # --- Reward: simple ranking by mode, small step penalty
        reward = self.cfg.mode_rewards[mode] - self.cfg.step_penalty

        # Finish bonus (unchanged)
        done = False
        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            done = True
        if self._steps >= self.cfg.max_steps:
            done = True

        # --- Blocking logic
        if delta > 0:
            # success -> reset blocks
            self._reset_blocks()
        else:
            if mode == 3:
                # backtrack used -> reset to start of cycle
                self._reset_blocks()
            else:
                # unsuccessful non-backtrack -> block this mode
                self._blocked[mode] = True

        # After-backtrack flag (kept for your obs if needed)
        self._after_backtrack = (mode == 3 and self.num_actions == 4)

        info = {
            "crossings": c_after,
            "delta": delta,
            "mode_requested": mode_req,
            "mode_effective": mode,
            "cap": cap,
            "blocked": tuple(self._blocked),
        }
        return self._obs(), reward, done, False, info

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.logger import configure

# === Paths ===
BASE = "/content/drive/MyDrive/crossing-reduction-hard"
PD_PATH = f"{BASE}/3-16.txt"
SMALL_JONES = f"{BASE}/small-jones.txt"
BEST_MODEL_PATH = f"{BASE}/best_model.zip"   # flat, no subfolder
TB_DIR = f"{BASE}/tb_knots"                  # tensorboard logs here
LOCAL_RANDOM = f"{BASE}/random_diagrams.csv"
LOCAL_RANDOM2 = f"{BASE}/random_diagrams_2.csv"
LOCAL3 = f"{BASE}/very_hard_unknots_2.csv"
os.makedirs(TB_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --------- 6) Load mixed CSVs from GCS (first column only), then shuffle ----------
# Files:
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"         # large file
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"
GCS_CSV_PATH_RANDOM = "gs://gdm-unknotting/random_diagrams.csv"    # much smaller file
GCS_CSV_PATH_RANDOM2 = "gs://gdm-unknotting/random_diagrams_2.csv"

# Parsing options (set per file if needed)
HAS_HEADER_MAIN = True
HAS_HEADER_VERY = True
HAS_HEADER_RANDOM = True
DELIMITER_MAIN  = None   # None = auto-sniff; else set "," or "\t"
DELIMITER_VERY  = None
DELIMITER_RANDOM  = None
MAX_ROWS_MAIN   = 385   # cap from the big file
SEED            = 42        # for reproducible shuffle

# 1) Read first column from all CSVs
raw_main = read_first_column_gcs(
    GCS_CSV_PATH_MAIN,
    max_lines=MAX_ROWS_MAIN,
    has_header=HAS_HEADER_MAIN,
    delimiter=DELIMITER_MAIN,
    encoding="utf-8",
)
raw_very = read_first_column_gcs(
    GCS_CSV_PATH_VERY,
    max_lines=None,                 # load the small file completely
    has_header=HAS_HEADER_VERY,
    delimiter=DELIMITER_VERY,
    encoding="utf-8",
)

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

raw_random = read_first_col_local(LOCAL_RANDOM, has_header=HAS_HEADER_RANDOM, encoding="utf-8")
raw_random2 = read_first_col_local(LOCAL_RANDOM2, has_header=HAS_HEADER_RANDOM, encoding="utf-8")
raw3 = read_first_col_local(LOCAL3, has_header=HAS_HEADER_RANDOM, encoding="utf-8")

print(f"[MAIN @ GCS] {len(raw_main)} rows (capped {MAX_ROWS_MAIN}).")
print(f"[VERY @ GCS] {len(raw_very)} rows.")
print(f"[RANDOM @ local] {len(raw_random)} rows.")
print(f"[RANDOM2 @ local] {len(raw_random2)} rows.")
print(f"[RAW3 @ local] {len(raw3)} rows.")

# 2) Concatenate (VERY_HARD first so they are guaranteed included), then clean to PD/DT
combined_raw = list(raw_very) + list(raw_main) + list(raw_random) + list(raw_random2) + list(raw3)
pd_lines = clean_pd_lines(combined_raw, max_keep=None)
assert pd_lines, "No valid PD/DT lines after cleaning—check the CSV columns or delimiters."

# 3) Shuffle for training (reproducible)
rng = random.Random(SEED)
rng.shuffle(pd_lines)

print(f"[Data] Kept {len(pd_lines)} PD/DT strings after strict filtering and shuffling.")
# (Optionally, deduplicate; uncomment if needed)
# pd_lines = list(dict.fromkeys(pd_lines))
# print(f"[Data] After de-duplication: {len(pd_lines)} PD/DT strings.")

#pd_lines_train = pd_lines[:TRAIN_ROWS]
#pd_lines_test = pd_lines[TRAIN_ROWS+1:MAX_ROWS]

def _obs_patch(self):
    c = crossings(self.L)
    try:
        comps = len(self.L.link_components)
    except Exception:
        comps = 1
    tmp = Link(self.L.PD_code())
    reduced = tmp.simplify(mode="basic")
    can_reduce = 1.0 if (reduced and crossings(tmp) < c) else 0.0
    recent = 1.0 if getattr(self, "_last_drop", 0) > 0 else 0.0
    return np.array([c, comps, self._steps, can_reduce, recent, 1.0], dtype=np.float32)

# Monkey-patch if needed
SphKnotEnv._obs = getattr(SphKnotEnv, "_obs", _obs_patch)

# --------- 7) Build envs ----------
# (Optional) If you want backtrack available in this run, set allow_backtrack=True here:
cfg = EnvCfg(max_steps=maxstepsdone, allow_backtrack=True)  # flip to True later if helpful
def make_env():
    return SphKnotEnv(pd_lines, cfg)

env = make_env()
vec_env = DummyVecEnv([lambda: make_env()])

obs, info = env.reset()
print("[Env] obs:", obs, "start crossings:", info["crossings"])

[MAIN @ GCS] 385 rows (capped 385).
[VERY @ GCS] 385 rows.
[RANDOM @ local] 770 rows.
[RANDOM2 @ local] 770 rows.
[RAW3 @ local] 385 rows.
[Data] Kept 2695 PD/DT strings after strict filtering and shuffling.
[Env] obs: [35.  1.  0.  0.  0.  1.] start crossings: 35


In [ ]:
# --------- 7) Train PPO ----------

# === Envs ===
vec_env  = DummyVecEnv([lambda: make_env()])
eval_env = DummyVecEnv([lambda: make_env()])

seed = random.randint(1, 100)
SEED = random.seed(seed)
TOTAL_STEPS = 12288*1
##################
##################
##################
##################

# === Load existing model or start fresh ===
if os.path.exists(BEST_MODEL_PATH):
    print(f"[Resume] Loading {BEST_MODEL_PATH}")
    model = PPO.load(BEST_MODEL_PATH, env=vec_env, device="auto")
    model.set_logger(configure(TB_DIR, ["stdout", "tensorboard"]))
else:
    print("[Fresh] Starting new PPO model")
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=lambda p: 3e-4 * p, # linear decay
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        tensorboard_log=TB_DIR,
        seed=SEED,
    )

# === Callbacks ===
# EvalCallback will now save directly as /MyDrive/crossing-reduction/best_model.zip
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=BASE,
    eval_freq=10000,
    n_eval_episodes=1,
    deterministic=True,
)
# (optional) periodic checkpoints — also in flat folder
ckpt_cb = CheckpointCallback(save_freq=50000, save_path=BASE, name_prefix="ppo")

# === Train (continues if loaded) ===
model.learn(
    total_timesteps=TOTAL_STEPS,
    callback=[eval_cb, ckpt_cb],
    reset_num_timesteps=False,
    progress_bar=True,
)

# === Save snapshot ===
final_path = f"{BASE}/ppo_knot_rl_spherogram_continued.zip"
model.save(final_path)
print(f"[Train] Saved -> {final_path}")

[Resume] Loading /content/drive/MyDrive/crossing-reduction-hard/best_model.zip
Logging to /content/drive/MyDrive/crossing-reduction-hard/tb_knots


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 656      |
|    ep_rew_mean     | 1.13e+03 |
| time/              |          |
|    fps             | 168      |
|    iterations      | 1        |
|    time_elapsed    | 12       |
|    total_timesteps | 122048   |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 656          |
|    ep_rew_mean          | 1.13e+03     |
| time/                   |              |
|    fps                  | 214          |
|    iterations           | 2            |
|    time_elapsed         | 19           |
|    total_timesteps      | 124096       |
| train/                  |              |
|    approx_kl            | 0.0005126436 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -3.5         |
|    explained_variance   | 7.03e-06     |
|    learning_r

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation 
environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and 
rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(

...

Episode length: 89.00 +/- 0.00

------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 89           |
|    mean_reward          | 175          |
| time/                   |              |
|    total_timesteps      | 130000       |
| train/                  |              |
|    approx_kl            | 9.655961e-05 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -3.47        |
|    explained_variance   | -1.03e-05    |
|    learning_rate        | 9.29e-06     |
|    loss                 | 814          |
|    n_updates            | 610          |
|    policy_gradient_loss | -0.000562    |
|    value_loss           | 1.66e+03     |
------------------------------------------


New best mean reward!

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 656      |
|    ep_rew_mean     | 1.13e+03 |
| time/              |          |
|    fps             | 222      |
|    iterations      | 5        |
|    time_elapsed    | 46       |
|    total_timesteps | 130240   |
---------------------------------
--------------------------------------------
| rollout/                |                |
|    ep_len_mean          | 656            |
|    ep_rew_mean          | 1.13e+03       |
| time/                   |                |
|    fps                  | 229            |
|    iterations           | 6              |
|    time_elapsed         | 53             |
|    total_timesteps      | 132288         |
| train/                  |                |
|    approx_kl            | 0.000101508165 |
|    clip_fraction        | 0              |
|    clip_range           | 0.2            |
|    entropy_loss         | -3.53          |
|    explained_variance   | -1

[Train] Saved -> /content/drive/MyDrive/crossing-reduction-hard/ppo_knot_rl_spherogram_continued.zip


In [ ]:
# ============================================================
# Evaluate ALL 385 instances at budget T=500, repeat 10 times,
# show ONE progress bar, and output:
#   - per-run % resolved (min_crossings==0)
#   - overall stats across 10 runs
#   - which instances are NEVER resolved (across all 10 runs)
#   - which instances are SOMETIMES resolved
#
# Assumes:
#   - pd_lines_test is your list of 385 PD/DT strings
#   - make_env(), parse_link_strict(), model, run_episode_detailed exist
#     (we reuse run_episode_detailed from earlier; it returns min_crossings, reached_zero, etc.)
# ============================================================

def eval_all_385_repeated(
    pd_lines,
    T: int = 500,
    n_repeats: int = 10,
    base_seed: int = 2026,
    deterministic: bool = True,
):
    n = len(pd_lines)
    assert n > 0

    # resolved_matrix[r, i] = whether repeat r resolved instance i
    resolved_matrix = np.zeros((n_repeats, n), dtype=bool)
    min_matrix      = np.full((n_repeats, n), fill_value=-1, dtype=int)

    total_jobs = n_repeats * n
    pbar = tqdm(total=total_jobs, desc=f"Evaluating {n} instances × {n_repeats} runs (T={T})", leave=True)

    per_run_summary = []

    for r in range(n_repeats):
        resolved_count = 0
        bestmins = []

        for i, line in enumerate(pd_lines):
            seed = base_seed + 100000 * r + i  # stable mapping: repeat r, instance i
            out = run_episode_detailed(line, T=T, seed=seed, deterministic=deterministic, log_traj=False)

            reached0 = bool(out["reached_zero"])
            mmin = int(out["min_crossings"])

            resolved_matrix[r, i] = reached0
            min_matrix[r, i]      = mmin

            resolved_count += int(reached0)
            bestmins.append(mmin)

            # update progress bar
            pbar.update(1)
            if (i % 10) == 0 or i == n - 1:
                pbar.set_postfix(run=r+1, resolved=f"{resolved_count}/{i+1} ({100*resolved_count/(i+1):.1f}%)", last_min=mmin)

        per_run_summary.append({
            "run": r + 1,
            "resolved_count": resolved_count,
            "resolved_pct": 100.0 * resolved_count / n,
            "median_min_crossings": float(np.median(bestmins)),
            "max_min_crossings": int(np.max(bestmins)),
        })

    pbar.close()

    df_runs = pd.DataFrame(per_run_summary)

    # Aggregate across repeats
    resolved_times = resolved_matrix.sum(axis=0)  # per instance: how many times resolved
    never_resolved_idx    = np.where(resolved_times == 0)[0].tolist()
    sometimes_resolved_idx= np.where((resolved_times > 0) & (resolved_times < n_repeats))[0].tolist()
    always_resolved_idx   = np.where(resolved_times == n_repeats)[0].tolist()

    summary = {
        "n_instances": n,
        "T": T,
        "n_repeats": n_repeats,
        "per_run_resolved_pct_mean": float(df_runs["resolved_pct"].mean()),
        "per_run_resolved_pct_std": float(df_runs["resolved_pct"].std(ddof=1)) if n_repeats > 1 else 0.0,
        "never_resolved_count": len(never_resolved_idx),
        "sometimes_resolved_count": len(sometimes_resolved_idx),
        "always_resolved_count": len(always_resolved_idx),
    }

    # Construct lists of the actual PD/DT strings (so you can save or inspect)
    never_resolved_lines     = [pd_lines[i] for i in never_resolved_idx]
    sometimes_resolved_lines = [pd_lines[i] for i in sometimes_resolved_idx]

    # A nice per-instance table
    df_inst = pd.DataFrame({
        "instance": np.arange(n),
        "resolved_times": resolved_times,
        "resolved_pct": 100.0 * resolved_times / n_repeats,
        "min_overall": min_matrix.min(axis=0),
        "min_median": np.median(min_matrix, axis=0),
        "min_max": min_matrix.max(axis=0),
    }).sort_values(["resolved_times", "min_overall"], ascending=[True, True])

    return df_runs, df_inst, summary, never_resolved_lines, sometimes_resolved_lines

# === Run it ===
df_runs, df_inst, summary, never_resolved_lines, sometimes_resolved_lines = eval_all_385_repeated(
    pd_lines_test,
    T=500,
    n_repeats=10,
    base_seed=2026,
    deterministic=True,
)

print("=== Summary across runs ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n=== Per-run resolved % ===")
display(df_runs)

print("\n=== Instances sorted by 'hardness' (least often resolved first) ===")
display(df_inst.head(30))

print("\n=== Never resolved (PD/DT strings) ===")
print(f"Count: {len(never_resolved_lines)}")
for s in never_resolved_lines[:50]:
    print(s)
if len(never_resolved_lines) > 50:
    print("... (truncated)")

print("\n=== Sometimes resolved (PD/DT strings) ===")
print(f"Count: {len(sometimes_resolved_lines)}")
for s in sometimes_resolved_lines[:50]:
    print(s)
if len(sometimes_resolved_lines) > 50:
    print("... (truncated)")

# Optional: save the unresolved lists
# with open("never_resolved.txt","w") as f: f.write("\n".join(never_resolved_lines))
# with open("sometimes_resolved.txt","w") as f: f.write("\n".join(sometimes_resolved_lines))
# df_inst.to_csv("per_instance_stats.csv", index=False)
# df_runs.to_csv("per_run_stats.csv", index=False)

Evaluating 385 instances × 10 runs (T=500):   0%|          | 0/3850 [00:00<?, ?it/s]

=== Summary across runs ===
n_instances: 385
T: 500
n_repeats: 10
per_run_resolved_pct_mean: 94.57142857142858
per_run_resolved_pct_std: 1.145019599176558
never_resolved_count: 0
sometimes_resolved_count: 119
always_resolved_count: 266

=== Per-run resolved % ===


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,run,resolved_count,resolved_pct,median_min_crossings,max_min_crossings
0,1,367,95.324675,0.0,46
1,2,367,95.324675,0.0,38
2,3,365,94.805195,0.0,39
3,4,357,92.727273,0.0,46
4,5,357,92.727273,0.0,41
5,6,360,93.506494,0.0,46
6,7,367,95.324675,0.0,39
7,8,368,95.584416,0.0,46
8,9,365,94.805195,0.0,46
9,10,368,95.584416,0.0,40



=== Instances sorted by 'hardness' (least often resolved first) ===


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,instance,resolved_times,resolved_pct,min_overall,min_median,min_max
377,377,4,40.0,0,26.0,28
19,19,5,50.0,0,18.5,39
45,45,5,50.0,0,14.0,28
224,224,5,50.0,0,23.0,46
238,238,5,50.0,0,16.5,35
197,197,6,60.0,0,0.0,38
256,256,6,60.0,0,0.0,37
271,271,6,60.0,0,0.0,27
285,285,6,60.0,0,0.0,37
333,333,6,60.0,0,0.0,31



=== Never resolved (PD/DT strings) ===
Count: 0

=== Sometimes resolved (PD/DT strings) ===
Count: 119
[[1, 35, 2, 34], [2, 26, 3, 25], [3, 52, 4, 53], [53, 4, 54, 5], [5, 51, 6, 50], [6, 21, 7, 22], [10, 8, 11, 7], [8, 16, 9, 15], [16, 10, 17, 9], [11, 30, 12, 31], [31, 12, 32, 13], [13, 32, 14, 33], [29, 14, 30, 15], [17, 39, 18, 38], [39, 19, 40, 18], [19, 41, 20, 40], [41, 21, 42, 20], [49, 23, 50, 22], [23, 46, 24, 47], [47, 24, 48, 25], [55, 26, 56, 27], [42, 27, 43, 28], [28, 37, 29, 38], [33, 44, 34, 45], [35, 1, 36, 56], [36, 44, 37, 43], [45, 48, 46, 49], [51, 54, 52, 55]]
[[1, 45, 2, 44], [2, 23, 3, 24], [3, 21, 4, 20], [53, 4, 54, 5], [5, 54, 6, 55], [6, 21, 7, 22], [22, 7, 23, 8], [55, 8, 56, 9], [46, 10, 47, 9], [35, 10, 36, 11], [11, 34, 12, 35], [47, 12, 48, 13], [52, 14, 53, 13], [27, 14, 28, 15], [15, 40, 16, 41], [29, 17, 30, 16], [17, 31, 18, 30], [18, 44, 19, 43], [19, 24, 20, 25], [42, 26, 43, 25], [26, 42, 27, 41], [39, 29, 40, 28], [60, 31, 61, 32], [67, 33, 68